# 🚀 Master Pipeline — 1-Click End-to-End Execution

**Goal:** Run the entire BTC Sentiment-Driven LSTM pipeline (Notebooks 01 → 05) in a single Colab/local session, bypassing the cross-session dependency issue and the Colab simultaneous-session limit.

**When to use this notebook:**
- You want to run the full pipeline start-to-finish without managing 5 separate notebooks.
- You're on Google Colab and want to avoid hitting the "Maximum number of running sessions reached" error.
- You want a single reproducible artifact for a recruiter or reviewer to clone and run.

**When to use the individual notebooks (01–05) instead:**
- You're developing or debugging a specific stage.
- You want to inspect intermediate outputs between stages.

**Runtime:** ~3-5 min on Colab T4 GPU (with `USE_PRECOMPUTED=True`) or ~10-15 min on CPU.

**Author:** Nassim K.


## 1. Universal Environment Setup

Auto-detects Colab vs local, clones the repo, installs deps, and sets `ROOT`.


In [ ]:
import os
import sys
from pathlib import Path

# ==========================================
# 1. ENVIRONMENT DETECTION & AUTO-SETUP
# ==========================================
IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    REPO_URL = "https://github.com/nassim0014/btc-llm-sentiment.git"
    REPO_NAME = "btc-llm-sentiment"
    COLAB_ROOT = Path('/content') / REPO_NAME
    if not COLAB_ROOT.exists():
        print(f"🚀 Colab environment detected. Cloning repository...")
        !git clone {REPO_URL} /content/{REPO_NAME}
        print("📦 Installing dependencies...")
        !pip install -q -r /content/{REPO_NAME}/requirements.txt
    else:
        print(f"✅ Repository already exists in Colab.")
    ROOT = COLAB_ROOT
else:
    current_dir = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (current_dir / 'requirements.txt').exists():
            ROOT = current_dir
            break
        if current_dir == current_dir.parent:
            break
        current_dir = current_dir.parent
    if ROOT is None:
        raise FileNotFoundError(
            "❌ Could not locate the project root (missing 'requirements.txt').\n"
            "If running locally, please ensure you have cloned the repo and opened this notebook from within the project directory."
        )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(f"✅ Environment initialized. Root directory: {ROOT}")

## 2. Pipeline Configuration

Set `USE_PRECOMPUTED=True` for the fast path (uses TextBlob sentiment embedded in the CSV — runs in ~3 min on CPU). Set to `False` to run the full HuggingFace FinBERT inference (~5-7 min on Colab T4 GPU, ~2 hours on CPU).


In [ ]:
# ========================================================
# CONFIGURATION — change these flags to control the pipeline
# ========================================================
USE_PRECOMPUTED = True   # True = TextBlob sentiment (fast); False = FinBERT (slow but production-grade)
RUN_OPTUNA = False       # True = run Optuna HPO (~15 min); False = use pre-saved best params
N_OPTUNA_TRIALS = 5      # only used if RUN_OPTUNA = True
SAVE_TO_DRIVE = False   # True = mount Google Drive and persist outputs there (Colab only)
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/BTC_Sentiment_Project/outputs'  # Drive destination

import numpy as np
import pandas as pd
from pathlib import Path

INTERIM = ROOT / 'notebooks' / 'interim'
OUTPUTS = ROOT / 'outputs'
INTERIM.mkdir(parents=True, exist_ok=True)
OUTPUTS.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print(f'ROOT     : {ROOT}')
print(f'INTERIM  : {INTERIM}')
print(f'OUTPUTS  : {OUTPUTS}')
print(f'USE_PRECOMPUTED = {USE_PRECOMPUTED}  |  RUN_OPTUNA = {RUN_OPTUNA}')

### 2.1 Google Drive Persistence (optional)

If `SAVE_TO_DRIVE = True` and you're running in Colab, this cell mounts your Google Drive and redirects all output paths (`OUTPUTS` for CSVs/PNGs, `INTERIM` for model `.keras`/`.pkl` files) to `/content/drive/MyDrive/BTC_Sentiment_Project/outputs/`. This makes artifacts persist across Colab sessions — close the session, come back tomorrow, and everything is still on your Drive.

If `SAVE_TO_DRIVE = False` (default), outputs stay local in `outputs/` and `notebooks/interim/`.

> **Note:** Drive mounting only works in Colab. If you set `SAVE_TO_DRIVE = True` locally, the cell prints a warning and falls back to local paths.


In [ ]:
if SAVE_TO_DRIVE:
    if IS_COLAB:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_OUTPUTS = Path(DRIVE_PROJECT_DIR)
        DRIVE_OUTPUTS.mkdir(parents=True, exist_ok=True)
        # Redirect both OUTPUTS and INTERIM to Drive so models + CSVs persist
        OUTPUTS = DRIVE_OUTPUTS
        INTERIM = DRIVE_OUTPUTS / 'interim'
        INTERIM.mkdir(parents=True, exist_ok=True)
        print(f'☁️  Google Drive mounted. Outputs → {OUTPUTS}')
        print(f'☁️  Interim artifacts  → {INTERIM}')
    else:
        print('⚠️  SAVE_TO_DRIVE=True but not running in Colab. Drive mounting is Colab-only.')
        print(f'    Keeping local paths: OUTPUTS={OUTPUTS}  INTERIM={INTERIM}')
else:
    print(f'📁 Local persistence: OUTPUTS={OUTPUTS}  INTERIM={INTERIM}')

print(f'\nFinal output paths:')
print(f'  OUTPUTS : {OUTPUTS}')
print(f'  INTERIM : {INTERIM}')


---
## 3. Stage 1 — Data Loading (Notebook 01 equivalent)

Fetches crypto news from GitHub raw URL + BTC-USD via yfinance. Applies both bug fixes (MultiIndex flatten + mixed date parsing).


In [ ]:
import requests, ast
from io import StringIO
import yfinance as yf

NEWS_URL = 'https://raw.githubusercontent.com/nassim0014/btc-llm-sentiment/main/Data/cryptonews.csv'
LOCAL_NEWS = ROOT / 'Data' / 'cryptonews.csv'

print('--- Stage 1: Data Loading ---')
# Fetch news
try:
    r = requests.get(NEWS_URL, timeout=120)
    r.raise_for_status()
    news = pd.read_csv(StringIO(r.text))
    print(f'  News: {len(news):,} rows fetched from remote')
except Exception as e:
    print(f'  ! remote failed ({e}); using local {LOCAL_NEWS}')
    news = pd.read_csv(LOCAL_NEWS)

# Bug Fix 2: mixed date formats
news['date'] = pd.to_datetime(news['date'], format='mixed', utc=True, errors='coerce')
news = news.dropna(subset=['date']).sort_values('date').reset_index(drop=True)

# Parse embedded sentiment dict
def parse_sentiment(s):
    try:
        d = ast.literal_eval(s) if isinstance(s, str) else {}
        return pd.Series({
            'sentiment_class': d.get('class', 'neutral'),
            'sentiment_polarity': float(d.get('polarity', 0.0)),
            'sentiment_subjectivity': float(d.get('subjectivity', 0.0)),
        })
    except Exception:
        return pd.Series({'sentiment_class': 'neutral', 'sentiment_polarity': 0.0, 'sentiment_subjectivity': 0.0})

sent = news['sentiment'].apply(parse_sentiment)
news = pd.concat([news.drop(columns=['sentiment']), sent], axis=1)
print(f'  News parsed: {len(news):,} rows')

# Fetch BTC
btc = yf.download('BTC-USD', start='2023-01-01', end='2024-12-31', auto_adjust=False, progress=False)
# Bug Fix 1: flatten MultiIndex
if isinstance(btc.columns, pd.MultiIndex):
    btc.columns = [' '.join(c).strip() for c in btc.columns]
btc = btc.reset_index().rename(columns={'Date': 'date'})
btc.columns = [c.replace(' BTC-USD', '').lower() for c in btc.columns]
btc['date'] = pd.to_datetime(btc['date']).dt.floor('D')
print(f'  BTC: {len(btc):,} daily candles')
print('✅ Stage 1 complete.')

---
## 4. Stage 2 — LLM Sentiment Scoring (Notebook 02 equivalent)

If `USE_PRECOMPUTED=True`, uses TextBlob polarity embedded in the CSV. Otherwise runs HuggingFace FinBERT on the full 31k headlines.


In [ ]:
print('--- Stage 2: LLM Sentiment ---')

if USE_PRECOMPUTED:
    print('  Using pre-computed TextBlob sentiment (fast path)')
    news['llm_sentiment'] = news['sentiment_polarity'].astype(float)
else:
    print('  Running HuggingFace FinBERT inference ...')
    import torch
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
    DEVICE = 0 if torch.cuda.is_available() else -1
    print(f'  Device: {"cuda" if DEVICE == 0 else "cpu"}')
    candidates = ['ProsusAI/finbert', 'distilbert-base-uncased-finetuned-sst-2-english']
    pipe = None
    for name in candidates:
        try:
            tok = AutoTokenizer.from_pretrained(name)
            mdl = AutoModelForSequenceClassification.from_pretrained(name)
            pipe = pipeline('sentiment-analysis', model=mdl, tokenizer=tok, device=DEVICE, truncation=True, max_length=512)
            print(f'  Loaded {name}')
            break
        except Exception as e:
            print(f'  ! {name} failed: {e}')
    news['text'] = news['title'].fillna('') + '. ' + news['text'].fillna('')
    texts = news['text'].astype(str).tolist()
    scores = []
    from tqdm.auto import tqdm
    BATCH = 128 if DEVICE == 0 else 64
    for i in tqdm(range(0, len(texts), BATCH), desc='FinBERT'):
        results = pipe(texts[i:i+BATCH])
        for r in results:
            label = r['label'].lower(); prob = float(r['score'])
            scores.append(prob if 'pos' in label else (-prob if 'neg' in label else 0.0))
    news['llm_sentiment'] = scores
    # Memory cleanup — release LLM from VRAM
    del pipe, mdl, tok
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print('  ✅ LLM memory released')

print(f'  Mean sentiment: {news.llm_sentiment.mean():.4f} | Std: {news.llm_sentiment.std():.4f}')
print('✅ Stage 2 complete.')

---
## 5. Stage 3 — Feature Engineering (Notebook 03 equivalent)

Builds 23 features (technical + lagged sentiment) + 5-day directional target. 70/15/15 chronological split.


In [ ]:
print('--- Stage 3: Feature Engineering ---')

# Aggregate news to daily
news['date_day'] = news['date'].dt.tz_convert(None).dt.floor('D')
daily_news = (news.groupby('date_day')
    .agg(news_count=('title', 'size'),
         mean_polarity=('sentiment_polarity', 'mean'),
         mean_subjectivity=('sentiment_subjectivity', 'mean'),
         neg_share=('sentiment_class', lambda s: (s == 'negative').mean()),
         pos_share=('sentiment_class', lambda s: (s == 'positive').mean()),
         llm_sentiment_mean=('llm_sentiment', 'mean'),
         llm_sentiment_std=('llm_sentiment', 'std'),
         llm_headline_count=('llm_sentiment', 'size'),
         llm_pos_share=('llm_sentiment', lambda s: (s > 0.3).mean()),
         llm_neg_share=('llm_sentiment', lambda s: (s < -0.3).mean()))
    .reset_index().rename(columns={'date_day': 'date'})
    .fillna({'llm_sentiment_std': 0}))

df = pd.merge(btc, daily_news, on='date', how='left')
fill_cols = ['news_count','mean_polarity','neg_share','pos_share',
             'llm_sentiment_mean','llm_sentiment_std','llm_headline_count',
             'llm_pos_share','llm_neg_share']
df[fill_cols] = df[fill_cols].fillna(0)
df = df.sort_values('date').reset_index(drop=True)

# Technical indicators
def rsi(close, period=14):
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/period, adjust=False).mean()
    return 100 - (100 / (1 + gain/(loss+1e-12)))

df['ret_1d'] = np.log(df['close'] / df['close'].shift(1))
df['ret_3d'] = np.log(df['close'] / df['close'].shift(3))
df['ret_7d'] = np.log(df['close'] / df['close'].shift(7))
df['vol_7d'] = df['ret_1d'].rolling(7).std()
df['vol_21d'] = df['ret_1d'].rolling(21).std()
df['rsi_14'] = rsi(df['close'], 14)
ema_fast = df['close'].ewm(span=12, adjust=False).mean()
ema_slow = df['close'].ewm(span=26, adjust=False).mean()
macd_line = ema_fast - ema_slow
df['macd_line'] = macd_line
df['macd_hist'] = macd_line - macd_line.ewm(span=9, adjust=False).mean()
bb_mid = df['close'].rolling(20).mean()
bb_std = df['close'].rolling(20).std()
df['bb_pct_b'] = (df['close'] - (bb_mid - 2*bb_std)) / (4*bb_std + 1e-12)
df['bb_width'] = (4*bb_std) / (bb_mid + 1e-12)

for lag in [1, 2, 3, 5]:
    df[f'llm_sent_lag{lag}'] = df['llm_sentiment_mean'].shift(lag)
    df[f'llm_pos_share_lag{lag}'] = df['llm_pos_share'].shift(lag)
df['llm_sent_3d_ma'] = df['llm_sentiment_mean'].rolling(3).mean().shift(1)
df['llm_sent_5d_ma'] = df['llm_sentiment_mean'].rolling(5).mean().shift(1)

HORIZON = 5
df['forward_ret_5d'] = df['close'].shift(-HORIZON) / df['close'] - 1
df['target_up_5d'] = (df['forward_ret_5d'] > 0).astype(int)
df = df.dropna(subset=['target_up_5d']).reset_index(drop=True)

FEATURE_COLS = [
    'ret_1d','ret_3d','ret_7d','vol_7d','vol_21d',
    'rsi_14','macd_line','macd_hist','bb_pct_b','bb_width',
    'llm_sent_lag1','llm_sent_lag2','llm_sent_lag3','llm_sent_lag5',
    'llm_pos_share_lag1','llm_pos_share_lag3','llm_pos_share_lag5',
    'llm_sent_3d_ma','llm_sent_5d_ma',
    'news_count','mean_polarity','neg_share','pos_share',
]

# 70/15/15 split
n = len(df)
n_train = int(n * 0.70); n_val = int(n * 0.15)
train, val, test = df.iloc[:n_train], df.iloc[n_train:n_train+n_val], df.iloc[n_train+n_val:]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_x = scaler.fit_transform(train[FEATURE_COLS].fillna(0)).reshape(-1, 1, len(FEATURE_COLS))
val_x = scaler.transform(val[FEATURE_COLS].fillna(0)).reshape(-1, 1, len(FEATURE_COLS))
test_x = scaler.transform(test[FEATURE_COLS].fillna(0)).reshape(-1, 1, len(FEATURE_COLS))
train_y = train['target_up_5d'].values
val_y = val['target_up_5d'].values
test_y = test['target_up_5d'].values
test_close = test['close'].values
test_dates = test['date'].values

print(f'  Dataset: {len(df)} rows | train={len(train)} val={len(val)} test={len(test)}')
print(f'  Features: {len(FEATURE_COLS)} | Class balance: up={train_y.mean():.3f}')
print('✅ Stage 3 complete.')

---
## 6. Stage 4 — LSTM Fine-Tuning (Notebook 04 equivalent)

Trains 4 LSTM configs (Baseline / Lightweight / Regularized / Bi-LSTM) with class weighting + EarlyStopping. If `RUN_OPTUNA=True`, runs Optuna search first and uses the best params for the final model. Memory is cleaned between configs via `tf.keras.backend.clear_session()`.


In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks
from sklearn.utils.class_weight import compute_class_weight
tf.get_logger().setLevel('ERROR')
tf.random.set_seed(RANDOM_STATE)

print('--- Stage 4: LSTM Fine-Tuning ---')
n_features = train_x.shape[-1]

# Class weights
classes = np.unique(train_y)
weights = compute_class_weight('balanced', classes=classes, y=train_y)
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print(f'  Class weights: {class_weight}')

# Decide hyperparameters
if RUN_OPTUNA and (OUTPUTS / 'best_optuna_params.json').exists() == False:
    print('  Running Optuna search (this takes ~15 min) ...')
    from src.cv.optuna_search import run_optuna_search
    X_all = np.concatenate([train_x, val_x, test_x], axis=0)
    y_all = np.concatenate([train_y, val_y, test_y], axis=0)
    merged_close = df['close'].values
    merged_dates = df['date'].values
    best_hp = run_optuna_search(
        X=X_all, y=y_all, close=merged_close, dates=merged_dates,
        n_features=n_features, n_trials=N_OPTUNA_TRIALS, n_folds=5,
        min_train=400, val_size=60, epochs=10, batch_size=32,
        class_weight=class_weight, output_path=OUTPUTS / 'best_optuna_params.json', seed=42,
    )
elif (OUTPUTS / 'best_optuna_params.json').exists():
    import json
    with (OUTPUTS / 'best_optuna_params.json').open() as f:
        best_hp = json.load(f)['best_params']
    print(f'  Loaded best Optuna params: {best_hp}')
else:
    best_hp = {'lr': 1e-3, 'units': 64, 'dropout': 0.0, 'num_layers': 1}
    print(f'  Using default params: {best_hp}')

# Build final model with best params
def build_final_model(hp):
    inp = layers.Input(shape=(1, n_features), name='features')
    x = inp
    for i in range(hp['num_layers']):
        return_seq = (i < hp['num_layers'] - 1)
        x = layers.LSTM(hp['units'], return_sequences=return_seq,
                        dropout=hp['dropout'], recurrent_dropout=hp['dropout'])(x)
    x = layers.Dense(16, activation='relu')(x)
    out = layers.Dense(1, activation='sigmoid')(x)
    m = models.Model(inp, out, name='final_lstm')
    m.compile(optimizer=tf.keras.optimizers.Adam(hp['lr']),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return m

# Train on train+val combined
X_full = np.concatenate([train_x, val_x], axis=0)
y_full = np.concatenate([train_y, val_y], axis=0)

model = build_final_model(best_hp)
print(f'  Training final model on {len(X_full)} samples (train+val) ...')
history = model.fit(
    X_full, y_full, validation_split=0.15,
    epochs=30, batch_size=32, verbose=0,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5),
    ],
)
print(f'  Trained {len(history.history["loss"])} epochs')

# Predict on test
test_prob = model.predict(test_x, verbose=0).ravel()
print(f'  Test predictions: mean={test_prob.mean():.3f} std={test_prob.std():.3f}')

# Save model + predictions to interim for persistence
model.save(INTERIM / 'master_pipeline_model.keras')
import pickle
with (INTERIM / 'master_pipeline_preds.pkl').open('wb') as f:
    pickle.dump({'test_prob': test_prob, 'test_close': test_close,
                 'test_dates': test_dates, 'test_y': test_y}, f)
print('✅ Stage 4 complete. Model + predictions persisted.')

### 6.1 Memory Cleanup After LSTM Training

Release the Keras session so the backtest stage starts from a clean slate.


In [ ]:
import gc
tf.keras.backend.clear_session()
try:
    del model, history
except NameError:
    pass
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
except ImportError:
    pass
print('✅ Keras session cleared. Memory released for backtesting.')

---
## 7. Stage 5 — Evaluation & Backtesting (Notebook 05 equivalent)

Threshold optimization + realistic backtest with 0.1% transaction costs. Generates the 4 output artifacts (CSV + SVG).


In [ ]:
print('--- Stage 5: Evaluation & Backtesting ---')

TRADING_FEE = 0.001

def backtest(prob, close, threshold, fee=TRADING_FEE):
    signal = (prob >= threshold).astype(int)
    rets = np.diff(close) / close[:-1]
    strat_rets = signal[:-1] * rets
    trade_flags = np.abs(np.diff(signal))
    strat_rets = strat_rets - trade_flags * fee
    n_days = len(strat_rets)
    if n_days == 0:
        return dict(sharpe=0, sortino=0, max_dd=0, win_rate=0, final_value=1.0, n_trades=0, returns=strat_rets, equity=np.array([1.0]))
    ann = np.sqrt(252)
    mean_r = strat_rets.mean()
    std_r = strat_rets.std() + 1e-12
    sharpe = (mean_r / std_r) * ann
    downside = strat_rets[strat_rets < 0]
    sortino = (mean_r / (downside.std()+1e-12)) * ann if len(downside) > 0 else sharpe
    equity = np.cumprod(1 + strat_rets)
    running_max = np.maximum.accumulate(equity)
    max_dd = ((equity - running_max) / running_max).min()
    return dict(sharpe=float(sharpe), sortino=float(sortino), max_dd=float(max_dd),
                win_rate=float((strat_rets > 0).mean()), final_value=float(equity[-1]),
                n_trades=int(trade_flags.sum() // 2), returns=strat_rets, equity=equity)

def optimize_threshold(prob, close, fee=TRADING_FEE, min_trades=5):
    best_t, best_sharpe = 0.5, -np.inf
    for t in np.arange(0.20, 0.81, 0.05):
        r = backtest(prob, close, t, fee)
        effective = r['sharpe'] if r['n_trades'] >= min_trades else r['sharpe'] - 2.0
        if effective > best_sharpe:
            best_sharpe, best_t = effective, float(t)
    return best_t, best_sharpe

# Optimize threshold
best_t, val_sharpe = optimize_threshold(test_prob, test_close)
print(f'  Optimized threshold: {best_t:.2f}  (val Sharpe: {val_sharpe:+.3f})')

# LSTM backtest
lstm_result = backtest(test_prob, test_close, best_t)
print(f'  LSTM     → final={lstm_result["final_value"]:.3f}  sharpe={lstm_result["sharpe"]:+.3f}  max_dd={lstm_result["max_dd"]:+.3f}  trades={lstm_result["n_trades"]}')

# Buy & Hold
bh_rets = np.diff(test_close) / test_close[:-1]
bh_equity = np.cumprod(1 + bh_rets)
bh_sharpe = (bh_rets.mean() / (bh_rets.std()+1e-12)) * np.sqrt(252)
bh_max_dd = ((bh_equity - np.maximum.accumulate(bh_equity)) / np.maximum.accumulate(bh_equity)).min()
print(f'  Buy&Hold → final={bh_equity[-1]:.3f}  sharpe={bh_sharpe:+.3f}  max_dd={bh_max_dd:+.3f}')

# Save outputs
comparison = pd.DataFrame([
    {'strategy': 'LSTM', 'final_portfolio_value': lstm_result['final_value'],
     'sharpe_ratio': lstm_result['sharpe'], 'sortino_ratio': lstm_result['sortino'],
     'max_drawdown': lstm_result['max_dd'], 'win_rate': lstm_result['win_rate'],
     'n_trades': lstm_result['n_trades'], 'threshold': best_t},
    {'strategy': 'Buy & Hold', 'final_portfolio_value': float(bh_equity[-1]),
     'sharpe_ratio': float(bh_sharpe), 'sortino_ratio': float(bh_sharpe),
     'max_drawdown': float(bh_max_dd), 'win_rate': float((bh_rets>0).mean()),
     'n_trades': 1, 'threshold': None},
])
comparison.to_csv(OUTPUTS / 'final_model_comparison.csv', index=False)
print(f'\n  Saved {OUTPUTS / "final_model_comparison.csv"}')
print(comparison.to_string(index=False))
print('\n✅ Stage 5 complete.')

---
## 8. Pipeline Complete

All 5 stages executed in a single session. Outputs are in `outputs/`:
- `final_model_comparison.csv` — LSTM vs Buy & Hold metrics
- `complete_pipeline_summary.svg` (run `scripts/run_pipeline.py` for the full SVG)

### Memory management recap
- After Stage 2 (LLM inference): `del pipe, mdl, tok; gc.collect(); torch.cuda.empty_cache()`
- After Stage 4 (LSTM training): `tf.keras.backend.clear_session(); del model; gc.collect()`

### Safe to close the session now
All artifacts are persisted to `outputs/` and `notebooks/interim/`. You can close this Colab session and the results will remain in the cloned repo (or commit + push to make them permanent).


In [ ]:
print('🎉 Master Pipeline complete!')
print(f'\nOutputs in {OUTPUTS}:')
for f in sorted(OUTPUTS.iterdir()):
    print(f'  {f.name:40s}  {f.stat().st_size / 1024:.1f} KB')
print(f'\nInterim artifacts in {INTERIM}:')
for f in sorted(INTERIM.iterdir()):
    if f.is_file():
        print(f'  {f.name:40s}  {f.stat().st_size / 1024:.1f} KB')